<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/RenameCalvingVideos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rename the calving videos and their output folders

One naming rule for every calving video **and** its folder in `CalvingPenPipelineOutputs`:

**`CalvingPen<pen><Mon><day>Cow<n>[ROI]`**: pen, month and day (no year, no leading zero), the recording number in brackets as the cow number (1 if none), `ROI` for cropped clips.

| Now | New name |
|---|---|
| `2026-11-11 00-00-00 Calving Pen 3 (2).avi` | `CalvingPen3Nov11Cow2.avi` |
| `2026-11-11 00-00-00 Calving Pen 3.avi` | `CalvingPen3Nov11Cow1.avi` |
| `2026-10-29 09-48-06 Calving Pen 2 (3).avi` | `CalvingPen2Oct29Cow3.avi` |
| `2026-09-26-CalvingPen3(1)_ROI_01-42-00_to_01-57-00_2FPS.avi` | `CalvingPen3Sep26Cow1ROI.avi` |
| output folder `2026-09-26 00-00-00 Calving Pen 2` | `CalvingPen2Sep26Cow1` |
| output folder `CalvingPen1Nov11Part1Win2` (automatic window) | `CalvingPen1Nov11Cow1Win2` |

**What is renamed:** video files in `CalvingPenVideos ` and `CalvingPenPipeline` (and their subfolders), and the video folders in `CalvingPenPipelineOutputs` (`Pool`, `PoolTest`, `Old` stay). Inside a renamed output folder the stored paths and names are updated too, so finished steps stay finished. Notebooks and spreadsheets keep their names.

**Safe:** cell 3 first **only shows** the plan (two tables: videos, output folders); nothing changes until `Apply = True`. Left alone: `Check` (a `Clip...` file or no date/pen in the name: rename by hand), `Unreadable` (broken shortcut or still uploading), `Conflict` (two files would get the same name, e.g. two ROI clips of one video, or the name is taken). Every rename goes into `CalvingPenPipelineOutputs/RenameLog.csv`; `Undo = True` puts the old names back. The pipeline finds a video by its old name (through the log) and by its new name before renaming.

**One person runs this, once, while nobody is running the pipeline** (a file renamed while a step reads it would fail). No GPU needed. Edit cell 2 and rerun; don't add cells.

In [1]:

# CELL 1: Mount Google Drive and establish project paths

import re
import os
import hashlib
import pandas as pd

from pathlib import Path
from datetime import datetime
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Main project directory
GroupDir = Path(
    "/content/drive/Shareddrives/BeastsAndBytes/ANSC4040GroupProject"
)

# Folders containing videos
Folders = ["CalvingPenVideos ", "CalvingPenPipeline"]

# Recognized video extensions
VideoTypes = {
    ".avi", ".mp4", ".mov", ".mkv", ".mpeg", ".mpg"
}

# Rename log
RenameLog = GroupDir / "CalvingPenPipelineOutputs" / "RenameLog.csv"

# Safety settings
Apply = True
Undo = False

# Check Drive mount
DriveRoot = Path("/content/drive")
print("Drive mounted:", DriveRoot.exists())

# Check project directory
print("\nProject directory:", GroupDir)
print("GroupDir exists:", GroupDir.exists())

if GroupDir.exists():
    print("\nContents of project directory:")
    for P in GroupDir.iterdir():
        print(repr(P.name), "| Directory:", P.is_dir())

    print("\nVideo folder checks:")
    for Folder in Folders:
        ExactPath = GroupDir / Folder
        StrippedPath = GroupDir / Folder.strip()

        Exists = ExactPath.exists() or StrippedPath.exists()
        print(repr(Folder), ":", Exists)

else:
    SharedDrives = DriveRoot / "Shareddrives"

    print("\nShared drives folder exists:", SharedDrives.exists())

    if SharedDrives.exists():
        print("Available shared drives:")
        for P in SharedDrives.iterdir():
            print(" -", repr(P.name))

print("\nRename log path:", RenameLog)
print("Apply:", Apply)
print("Undo:", Undo)


Mounted at /content/drive
Drive mounted: True

Project directory: /content/drive/Shareddrives/BeastsAndBytes/ANSC4040GroupProject
GroupDir exists: True

Contents of project directory:
'CalvingPenPipeline' | Directory: True
'CalvingPenPipelineOutputs' | Directory: True
'ParlorIDMatch' | Directory: True
'BoviSyncRecords' | Directory: True
'EntryParlorVideos' | Directory: True
'GroupLog.gdoc' | Directory: False
'2026-09-26-CalvingPen(2.2).avi' | Directory: False
'ANSC4040GroupProject-Justin.ipynb' | Directory: False
'NamingConvention' | Directory: False
'CalvingPenVideos ' | Directory: True

Video folder checks:
'CalvingPenVideos ' : True
'CalvingPenPipeline' : True

Rename log path: /content/drive/Shareddrives/BeastsAndBytes/ANSC4040GroupProject/CalvingPenPipelineOutputs/RenameLog.csv
Apply: True
Undo: False


In [4]:


# CELL 2: Naming functions and rename preview plan

# Safety settings
Apply = True
Undo = False
RenameOutputFolders = False

# Folder names inside the project directory
Folders = [
    "CalvingPenVideos",
    "CalvingPenPipeline"
]


def GetVideoInfo(OldName):
    """Extract date, pen, cow, clip, and ROI information."""

    Name = Path(OldName).stem

    # Extract date
    DateMatch = re.search(
        r"(\d{4})[-_](\d{2})[-_](\d{2})",
        Name
    )

    if not DateMatch:
        return None

    Year, Month, Day = map(int, DateMatch.groups())

    try:
        Date = datetime(Year, Month, Day)
    except ValueError:
        return None

    # Extract pen number
    PenMatch = re.search(
        r"Calving[\s_-]*Pen[\s_-]*(\d+)",
        Name,
        re.IGNORECASE
    )

    if not PenMatch:
        PenMatch = re.search(
            r"Pen[\s_-]*(\d+)",
            Name,
            re.IGNORECASE
        )

    if not PenMatch:
        return None

    Pen = PenMatch.group(1)

    # Identify clip files
    ClipMatch = re.search(
        r"Clip(\d+)",
        Name,
        re.IGNORECASE
    )

    # Clips are assigned Cow1
    if ClipMatch:
        Cow = 1
    else:
        CowMatch = re.search(r"\((\d+)\)", Name)
        Cow = int(CowMatch.group(1)) if CowMatch else 1

    # Extract ROI time window
    ROIMatch = re.search(
        r"ROI[_-]?(\d{2})-(\d{2})-(\d{2})"
        r"_to[_-](\d{2})-(\d{2})-(\d{2})",
        Name,
        re.IGNORECASE
    )

    ROITime = None

    if ROIMatch:
        ROITime = tuple(map(int, ROIMatch.groups()))

    IsROI = bool(
        re.search(r"ROI", Name, re.IGNORECASE)
    )

    return {
        "Date": Date,
        "Pen": Pen,
        "Cow": Cow,
        "Clip": ClipMatch.group(1) if ClipMatch else None,
        "IsROI": IsROI,
        "ROITime": ROITime,
        "MonthName": Date.strftime("%b"),
        "Day": Date.day
    }


def GenerateNewName(OldName, ROIIndex=None):
    """Generate the new standardized filename."""

    P = Path(OldName)
    Info = GetVideoInfo(OldName)

    if Info is None:
        return None

    Extension = P.suffix.lower()

    Base = (
        f"CalvingPen{Info['Pen']}"
        f"{Info['MonthName']}{Info['Day']}"
        f"Cow{Info['Cow']}"
    )

    # Clip files
    if Info["Clip"] is not None:
        ClipNumber = int(Info["Clip"])
        return f"{Base}Part{ClipNumber}{Extension}"

    # ROI files
    if Info["IsROI"]:

        if ROIIndex is None:

            # Special 2-minute ROI clip
            if re.search(r"\)ROI$", P.stem, re.IGNORECASE):
                ROIIndex = 3
            else:
                return None

        return f"{Base}ROI{ROIIndex}{Extension}"

    # Regular video
    return Base + Extension


def PlanRenames(Folders, Recursive=True):
    """Create a rename preview without changing any files."""

    Rows = []
    AllFiles = []

    # --------------------------------------------------
    # 1. Find all video files
    # --------------------------------------------------

    for Folder in Folders:

        Root = GroupDir / Folder

        # Automatically handle trailing spaces in folder names
        if not Root.exists():

            Matches = [
                P for P in GroupDir.iterdir()
                if P.is_dir()
                and P.name.strip() == Folder.strip()
            ]

            if len(Matches) == 1:
                Root = Matches[0]

            elif len(Matches) == 0:
                raise FileNotFoundError(
                    f"Could not find folder matching: {Folder}"
                )

            else:
                raise RuntimeError(
                    f"Multiple folders match: {Folder}: {Matches}"
                )

        print(f"Scanning: {Root}")

        if Recursive:
            Files = Root.rglob("*")
        else:
            Files = Root.iterdir()

        for P in Files:
            if (
                P.is_file()
                and P.suffix.lower() in VideoTypes
            ):
                AllFiles.append(P)

    print(f"\nFound {len(AllFiles)} video files.")

    # --------------------------------------------------
    # 2. Assign chronological ROI numbers
    # --------------------------------------------------

    ROIWindows = {}

    for P in AllFiles:

        Info = GetVideoInfo(P.name)

        if Info is None or not Info["IsROI"]:
            continue

        if Info["ROITime"] is None:
            continue

        Key = (
            Info["Date"].date(),
            Info["Pen"],
            Info["Cow"]
        )

        ROIWindows.setdefault(Key, set()).add(
            Info["ROITime"]
        )

    ROIIndexes = {}

    for Key, Windows in ROIWindows.items():

        SortedWindows = sorted(Windows)

        for Index, Window in enumerate(SortedWindows, start=1):
            ROIIndexes[(Key, Window)] = Index

    # --------------------------------------------------
    # 3. Identify possible duplicate ROI files
    # --------------------------------------------------

    DuplicateROIPaths = set()
    DuplicateROIWindows = {}

    for P in AllFiles:

        Info = GetVideoInfo(P.name)

        if (
            Info is None
            or not Info["IsROI"]
            or Info["ROITime"] is None
        ):
            continue

        Key = (
            str(P.parent),
            Info["Date"].date(),
            Info["Pen"],
            Info["Cow"],
            Info["ROITime"]
        )

        DuplicateROIWindows.setdefault(Key, []).append(P)

    for Key, FilesForWindow in DuplicateROIWindows.items():

        if len(FilesForWindow) > 1:

            # Prefer AVI when the same ROI window occurs
            # more than once in the same folder
            OrderedFiles = sorted(
                FilesForWindow,
                key=lambda P: (
                    P.suffix.lower() != ".avi",
                    P.name
                )
            )

            # Flag additional copies for review
            for P in OrderedFiles[1:]:
                DuplicateROIPaths.add(P)

    # --------------------------------------------------
    # 4. Create one preview row for each video
    # --------------------------------------------------

    for P in AllFiles:

        Info = GetVideoInfo(P.name)

        FolderName = str(
            P.parent.relative_to(GroupDir)
        )

        New = None
        Status = None
        ROIIndex = None

        # Identify the known duplicate original video
        IsDuplicateCopy = P.name.startswith(
            "Copy of 2026-09-26 00-00-00 Calving Pen 3"
        )

        if IsDuplicateCopy:

            New = None
            Status = "Duplicate - verify/delete"

        elif P in DuplicateROIPaths:

            New = None
            Status = "Possible duplicate ROI - review/delete"

        elif ".partial." in P.name.lower():

            New = None
            Status = "Check - incomplete file"

        elif Info is None:

            New = None
            Status = "Check - unrecognized filename"

        else:

            # Determine ROI number when applicable
            if Info["IsROI"]:

                if Info["ROITime"] is not None:

                    Key = (
                        Info["Date"].date(),
                        Info["Pen"],
                        Info["Cow"]
                    )

                    ROIIndex = ROIIndexes.get(
                        (Key, Info["ROITime"])
                    )

                elif re.search(
                    r"\)ROI$",
                    P.stem,
                    re.IGNORECASE
                ):

                    # Special 2-minute ROI clip
                    ROIIndex = 3

            # Generate the new name exactly once
            New = GenerateNewName(
                P.name,
                ROIIndex
            )

            if New is None:
                Status = "Check - missing ROI number"
            elif New == P.name:
                Status = "Same"
            else:
                Status = "Rename"

        # Check file accessibility
        try:
            P.stat()
        except OSError:
            Status = "Unreadable"

        Rows.append({
            "Folder": FolderName,
            "OldName": P.name,
            "NewName": New,
            "Status": Status
        })

    # --------------------------------------------------
    # 5. Build preview DataFrame
    # --------------------------------------------------

    Plan = pd.DataFrame(
        Rows,
        columns=[
            "Folder",
            "OldName",
            "NewName",
            "Status"
        ]
    )

    # --------------------------------------------------
    # 6. Detect duplicate destination names
    # --------------------------------------------------

    if len(Plan):

        Valid = (
            Plan["Status"].eq("Rename")
            & Plan["NewName"].notna()
        )

        Destination = (
            Plan["Folder"] + "/" + Plan["NewName"]
        )

        Twice = (
            Destination.duplicated(keep=False)
            & Valid
        )

        Taken = Plan.apply(
            lambda R: (
                R["Status"] == "Rename"
                and (
                    GroupDir
                    / R["Folder"]
                    / R["NewName"]
                ).exists()
                and R["OldName"] != R["NewName"]
            ),
            axis=1
        )

        Plan.loc[
            Twice | Taken,
            "Status"
        ] = "Conflict"

    return Plan


In [6]:
# CELL 3: Generate and review rename preview


# CELL 3: Preview and apply renames

Apply = True
Undo = False

Plan = PlanRenames(Folders)

print("\nRENAME SUMMARY")
print(Plan["Status"].value_counts().to_string())

print("\nPROPOSED FILENAMES")
display(
    Plan.sort_values(
        ["Folder", "OldName"]
    ).reset_index(drop=True)
)

# --------------------------------------------------
# APPLY RENAMES
# --------------------------------------------------

if Apply and not Undo:

    Eligible = Plan[
        Plan["Status"].eq("Rename")
    ]

    print(f"\nFiles eligible for renaming: {len(Eligible)}")

    Renamed = 0
    Skipped = 0

    for _, Row in Eligible.iterrows():

        Source = GroupDir / Row["Folder"] / Row["OldName"]
        Destination = GroupDir / Row["Folder"] / Row["NewName"]

        if not Source.exists():
            print(f"SKIPPED - Source missing: {Source.name}")
            Skipped += 1
            continue

        if Destination.exists():
            print(f"SKIPPED - Destination already exists: {Destination.name}")
            Skipped += 1
            continue

        try:
            Source.rename(Destination)
            print(f"RENAMED: {Source.name} -> {Destination.name}")
            Renamed += 1

        except Exception as E:
            print(f"ERROR: {Source.name}: {E}")
            Skipped += 1

    print("\nRENAME RESULTS")
    print(f"Successfully renamed: {Renamed}")
    print(f"Skipped: {Skipped}")

else:
    print("\nPREVIEW ONLY: No files have been renamed.")


Scanning: /content/drive/Shareddrives/BeastsAndBytes/ANSC4040GroupProject/CalvingPenVideos 
Scanning: /content/drive/Shareddrives/BeastsAndBytes/ANSC4040GroupProject/CalvingPenPipeline

Found 27 video files.

RENAME SUMMARY
Status
Rename                                    24
Duplicate - verify/delete                  1
Possible duplicate ROI - review/delete     1
Check - incomplete file                    1

PROPOSED FILENAMES


,Folder,OldName,NewName,Status
0,CalvingPenPipeline,2026-09-26-Clip1Pen2(4:37hrs).avi,CalvingPen2Sep26Cow1Part1.avi,Rename
1,CalvingPenPipeline,2026-09-26-Clip2Pen2(4:37hrs).avi,CalvingPen2Sep26Cow1Part2.avi,Rename
2,CalvingPenPipeline/Area cropping,2026-09-26-CalvingPen3(1)_ROI_01-42-00_to_01-5...,CalvingPen3Sep26Cow1ROI2.avi,Rename
3,CalvingPenPipeline/Area cropping,Copy of 2026-10-29 09-48-06 Calving Pen 2 (3)_...,CalvingPen2Oct29Cow3ROI1.avi,Rename
4,CalvingPenVideos,2026-09-26-CalvingPen1(1).avi,CalvingPen1Sep26Cow1.avi,Rename
5,CalvingPenVideos,2026-09-26-CalvingPen1(1).mp4,CalvingPen1Sep26Cow1.mp4,Rename
6,CalvingPenVideos,2026-09-26-CalvingPen2(1).avi,CalvingPen2Sep26Cow1.avi,Rename
7,CalvingPenVideos,2026-09-26-CalvingPen2(2).avi,CalvingPen2Sep26Cow2.avi,Rename
8,CalvingPenVideos,2026-09-26-CalvingPen2(2).mp4,CalvingPen2Sep26Cow2.mp4,Rename
9,CalvingPenVideos,2026-09-26-CalvingPen3(1).avi,CalvingPen3Sep26Cow1.avi,Rename



Files eligible for renaming: 24
RENAMED: 2026-09-26-CalvingPen2(2).avi -> CalvingPen2Sep26Cow2.avi
RENAMED: 2026-10-29 09-48-06 Calving Pen 2 (3).avi -> CalvingPen2Oct29Cow3.avi
RENAMED: 2026-10-29 09-48-06 Calving Pen 2 (6).avi -> CalvingPen2Oct29Cow6.avi
RENAMED: 2026-10-29_09-48-06_Calving_Pen_2_(3)ROI.mp4 -> CalvingPen2Oct29Cow3ROI3.mp4
RENAMED: 2026-09-26-CalvingPen3(2).avi -> CalvingPen3Sep26Cow2.avi
RENAMED: 2026-09-26-CalvingPen3(1).avi -> CalvingPen3Sep26Cow1.avi
RENAMED: 2026-09-26-CalvingPen1(1).avi -> CalvingPen1Sep26Cow1.avi
RENAMED: 2026-09-26-CalvingPen2(1).avi -> CalvingPen2Sep26Cow1.avi
RENAMED: 2026-10-29 09-48-06 Calving Pen 2 (3)_ROI_01-06-51_to_01-26-29_2FPS.avi -> CalvingPen2Oct29Cow3ROI1.avi
RENAMED: 2026-09-26-CalvingPen3(1)_ROI_01-42-00_to_01-57-00_2FPS.mp4 -> CalvingPen3Sep26Cow1ROI2.mp4
RENAMED: 2026-11-11 00-00-00 Calving Pen 1.avi -> CalvingPen1Nov11Cow1.avi
RENAMED: 2026-11-11 00-00-00 Calving Pen 1 (2).avi -> CalvingPen1Nov11Cow2.avi
RENAMED: 2026-11-11 

## About this notebook

**Rule** (`CalvingName` in `CalvingPen/CalvingPipeline/Rename.py`, from the group's naming notebook): `CalvingPen<pen><Mon><day>Cow<n>[ROI]`. The number in brackets after the pen (`Calving Pen 3 (2)`) is the cow number; without one it is `Cow1`. A start time, ROI time window or fps in the old name is dropped (only `ROI` is kept), so two ROI clips of one video get the same name and are left as `Conflict`. An automatic-window folder keeps its `Win1`, `Win2` ending.

**Status column:** `Rename` (will be renamed), `Same` (already right), `Check` (left alone: rename by hand), `Unreadable` (broken Drive shortcut or still uploading), `Conflict` (left alone).

**Afterwards:** use the new names in `PipelineSteps` (`VideoFile`); old names still find the video. Cow labels in the pool are `Pen2Cow1`, `Pen2Cow2`, ... per pen in date order (`CowRegister.csv`); a video's `Cow<n>` tells the pool which cow of that pen and day its tracks are.